In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

# Embedding Generation
The objective of this notebook is to convert document chunks into vector embeddings using a Sentence Transformer model.

Embeddings are numerical representations of text that capture semantic meaning. Similar pieces of text are represented by vectors that are close together in vector space.

These embeddings will later be stored in a FAISS vector database and used to retrieve relevant document chunks during question answering.

In [5]:
# Import libraries required for PDF extraction, chunking and embeddings

from pypdf import PdfReader

from langchain_text_splitters import RecursiveCharacterTextSplitter

from sentence_transformers import SentenceTransformer

In [7]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

print("Text splitter loaded successfully")

Text splitter loaded successfully


In [10]:
# Import libraries required for PDF extraction, chunking and embeddings

from pypdf import PdfReader

from langchain_text_splitters import RecursiveCharacterTextSplitter

from sentence_transformers import SentenceTransformer

In [11]:
# Load the General Academic Regulations document

pdf_path = "/kaggle/input/datasets/makhosazanemthethwa/cos802-university-documents/General_Academic_Regulations_2026.pdf"

reader = PdfReader(pdf_path)

In [12]:
# Extract text from all pages

full_text = ""

for page in reader.pages:

    page_text = page.extract_text()

    if page_text:
        full_text += page_text + "\n"

print(f"Characters: {len(full_text):,}")
print(f"Words: {len(full_text.split()):,}")

Characters: 361,882
Words: 53,391


In [13]:
# Recreate the chunking strategy validated in Notebook 02

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = text_splitter.split_text(full_text)

print(f"Chunks Created: {len(chunks)}")

Chunks Created: 455


In [14]:
# Load a lightweight Sentence Transformer model suitable for semantic retrieval

model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (3 outputs left out)

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [15]:
# Convert text chunks into vector embeddings

embeddings = model.encode(
    chunks,
    show_progress_bar=True
)

print(f"Embeddings Generated: {len(embeddings)}")

Batches:   0%|          | 0/15 [00:00<?, ?it/s]

Embeddings Generated: 455


In [16]:
# Display the size of an embedding vector

print(f"Embedding Dimension: {len(embeddings[0])}")


Embedding Dimension: 384


In [17]:
# Display the first few values of the first embedding

print(embeddings[0][:10])

[-0.04127716  0.07025339 -0.02427595 -0.05300611 -0.02989284  0.00507703
  0.00277067  0.01535071 -0.04664855  0.10182578]


## Observations
The General Academic Regulations document was successfully transformed into 455 text chunks and converted into vector embeddings using the all-MiniLM-L6-v2 Sentence Transformer model.

Each chunk was represented as a numerical vector capturing its semantic meaning. The embedding generation process completed successfully and produced a vector representation for every chunk.

These embeddings will form the basis of semantic retrieval in the next stage of the RAG pipeline.

## Conclusion
Embedding generation was successfully completed.

The chunked university document was transformed into dense vector representations suitable for semantic search and retrieval. These embeddings will be stored in a vector database in the next stage of the project.

The next notebook will focus on creating a FAISS vector store and testing semantic retrieval.